# 01seaice: Arctic sea ice for each day of the voyage

Daily sea ice concentration from the AMSR2 satellite radiometer, processed by the University of Bremen (ASI algorithm, 6.25 km grid). For every day from departure to return, this notebook:

1. downloads the day's GeoTIFF (skipping files already in `data/raw/seaice`),
2. turns the concentration map into ice shapes in three shades (for the map page, `index.html`),
3. paints the same ice as a transparent image (for the 3D page, `cesium.html`),
4. measures the ice extent, and how far the ship was from the nearest ice that day.

Input: `docs/data/route.json` from `00route`. Output: `docs/data/seaice.json` (map page), `docs/data/seaice_days.json` (3D page) and one image per day in `docs/data/ice/`.

**Data credit:** Spreen, G., L. Kaleschke and G. Heygster (2008), Sea ice remote sensing using AMSR-E 89 GHz channels, J. Geophys. Res. 113, C02S03, doi:10.1029/2005JC003384. Data from the University of Bremen, https://seaice.uni-bremen.de.

## Section 0: settings

Everything you may want to change is here, marked `# <-`.

Concentration is the share of the sea surface covered by ice, in percent. 15% is the usual cut-off for "ice" in sea ice extent statistics.

In [ ]:
ROUTE_FILE = "docs/data/route.json"  # <- from 00route (gives the dates and the ship's track)
RAW_DIR = "data/raw/seaice"  # <- where the daily GeoTIFFs are kept
OUT_FILE = "docs/data/seaice.json"  # <- everything, for the map page
DAYS_FILE = "docs/data/seaice_days.json"  # <- the daily numbers only, for the 3D page (which draws the ice from the images)

URL = "https://data.seaice.uni-bremen.de/amsr2/asi_daygrid_swath/n6250/{year}/{mon}/Arctic/asi-AMSR2-n6250-{ymd}-v5.4.tif"  # <-
FILE = "asi-AMSR2-n6250-{ymd}-v5.4.tif"  # <- file name pattern (keep as downloaded)

ICE_MIN = 15  # <- percent: counts as ice from here up
LEVELS = [15, 50, 80]  # <- the three ice shades on the globe: at least 15, 50 and 80 percent
LAND = 120  # <- value used for land (and the satellite's blind spot at the pole) in the files

POLE_HOLE_KM = 100  # <- the satellite cannot see this close to the pole; filled as fully ice covered
SMOOTH_CELLS = 3  # <- VISUAL: smooths the map over this many cells (3 = 3x3 cells) before drawing
SIMPLIFY_KM = 3  # <- VISUAL: simplifies the ice outlines (bigger = coarser, smaller file)
MIN_AREA_KM2 = 300  # <- VISUAL: drops ice patches smaller than this
POLE_CUT_LAT = 85  # <- the globe cannot draw shapes north of about 85 N; a disc covers the cap instead
SLIT_M = 2  # <- width of the cut along the 180 degree meridian, in metres (wide enough to split the shapes, too thin to see)
STEP_KM = 50  # <- extra points along long outline edges, so they keep their true path in longitude/latitude
IMG_DIR = "docs/data/ice"  # <- the daily images for the 3D page
IMG_DEG = 0.05  # <- image pixel size in degrees (smaller = sharper, bigger files)
IMG_SOUTH = 50  # <- the images cover the Arctic north of this latitude
IMG_COLORS = [(200, 225, 245, 89), (216, 236, 249, 189), (242, 248, 254, 245)]  # <- VISUAL: red, green, blue, opacity (0-255) of the three shades, as they look stacked on the map page
SOUTH_LIMIT = 55  # <- latitude: the ship is compared with the ice only north of this

## 1. Imports

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject, transform
from PIL import Image
import contourpy
from shapely import segmentize
from shapely.geometry import Polygon, MultiPolygon, box, mapping
from shapely.ops import unary_union
from shapely.validation import make_valid
import matplotlib.pyplot as plt

## 2. The days of the voyage

Day 0 is the departure date. The ice of each date is shown on the globe while the ship's clock is in that day.

In [ ]:
route = json.load(open(ROUTE_FILE))
dates = pd.date_range(route["depart"], route["return"], freq="D")
raw = Path(RAW_DIR)
raw.mkdir(parents=True, exist_ok=True)
print(f"{len(dates)} days, {dates[0].date()} to {dates[-1].date()}")

## 3. Download

Files already in `RAW_DIR` are skipped, so you can stop and run this cell again at any time. If a download fails, the cell carries on and lists the missing days at the end; you can also download those by hand into `RAW_DIR`.

In [ ]:
todo = [d for d in dates if not (raw / FILE.format(ymd=d.strftime("%Y%m%d"))).exists()]
print(f"{len(dates) - len(todo)} of {len(dates)} files already here, {len(todo)} to download")

start = time.time()
failed = []
for i, d in enumerate(todo, start=1):
    ymd = d.strftime("%Y%m%d")
    url = URL.format(year=d.year, mon=d.strftime("%b").lower(), ymd=ymd)
    target = raw / FILE.format(ymd=ymd)
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        part = target.with_suffix(".part")
        part.write_bytes(r.content)
        part.replace(target)
    except Exception as e:
        failed.append((ymd, str(e)[:80]))
    spent = time.time() - start
    left = spent / i * (len(todo) - i)
    bar = "#" * int(30 * i / len(todo))
    print(f"\r[{bar:<30}] {i}/{len(todo)}  {spent:4.0f} s so far, about {left:4.0f} s left", end="")
print()
if failed:
    print("Not downloaded:")
    for ymd, err in failed:
        print("  ", ymd, err)

## 4. Reading a day

The files use the NSIDC polar stereographic grid (EPSG:3413): every cell is 6.25 km across in the projection. Land and the satellite's blind spot around the pole both carry the value `LAND`; the blind spot (within `POLE_HOLE_KM` of the pole) is filled as fully ice covered, as is usual for this data.

The ice extent is the area of all cells with at least `ICE_MIN` percent ice. Cells away from 70 degrees north are slightly smaller in reality than in the projection, which the scale factor of the projection corrects.

In [ ]:
def read_day(path):
    """Concentration (percent, float) on the original grid; land is NaN."""
    with rasterio.open(path) as src:
        a = src.read(1).astype("float32")
        meta = (src.transform, src.crs, src.width, src.height)
    tr, crs, w, h = meta
    xs = tr.c + (np.arange(w) + 0.5) * tr.a
    ys = tr.f + (np.arange(h) + 0.5) * tr.e
    dist = np.hypot(*np.meshgrid(xs, ys)) / 1000
    a[(a > 100) & (dist < POLE_HOLE_KM)] = 100  # the pole hole
    a[a > 100] = np.nan  # land
    return a, meta


def cell_area_km2(meta):
    """True area of each cell (polar stereographic, true scale at 70 N)."""
    tr, crs, w, h = meta
    cols, rows = np.meshgrid(np.arange(w), np.arange(h))
    xs = tr.c + (cols + 0.5) * tr.a
    ys = tr.f + (rows + 0.5) * tr.e
    lon, lat = transform(crs, "EPSG:4326", xs.ravel().tolist(), ys.ravel().tolist())
    lat = np.radians(np.array(lat).reshape(h, w))
    k = (1 + np.sin(np.radians(70))) / (1 + np.sin(lat))  # scale factor on the sphere
    return (abs(tr.a) / 1000) * (abs(tr.e) / 1000) / k ** 2


def smooth(a, n):
    """Mean over n x n cells, ignoring land."""
    if n <= 1:
        return a
    h, w = a.shape
    r = n // 2
    pad = np.pad(a, r, constant_values=np.nan)
    stack = np.stack([pad[i:i + h, j:j + w] for i in range(n) for j in range(n)])
    with warnings.catch_warnings():  # cells with only land around them give an empty mean: harmless
        warnings.simplefilter("ignore", RuntimeWarning)
        out = np.nanmean(stack, axis=0)
    out[np.isnan(a)] = np.nan
    return out

## 5. Ice shapes for the globe

The shapes are drawn like isolines on a weather map: contours at each of the `LEVELS`, traced on the original grid with interpolation, so the outlines come out smooth. The three shades are nested (at least 15, at least 50, at least 80 percent), so they stack on the globe without gaps.

Three details make the shapes work on a longitude/latitude globe:

- **The date line:** a hair-thin slit (`SLIT_M`, 2 m) is cut from the pole along the 180 degree meridian, so no shape wraps around the pole.
- **The pole:** the globe cannot draw shapes north of about 85 N. Shapes stop at `POLE_CUT_LAT`, and the page covers the cap with one disc, shaded by the day's ice north of that latitude (`cap_level` below).
- **Long edges:** a straight edge on the polar grid (such as the sides of the slit) is a curve in longitude/latitude. Points are added every `STEP_KM` along each outline before converting, so every edge keeps its true path. Without them, an edge from 73 N to the pole was drawn as a diagonal and cut a wedge out of the ice.

In [ ]:
def make_slit(crs):
    (x1,), (y1,) = transform("EPSG:4326", crs, [180.0], [50.0])
    length = np.hypot(x1, y1)
    ux, uy = x1 / length, y1 / length
    px, py = -uy * SLIT_M / 2, ux * SLIT_M / 2
    far = length * 1.5
    return Polygon([(px, py), (px + ux * far, py + uy * far), (-px + ux * far, -py + uy * far), (-px, -py)])


def ice_shapes(a, meta, level, slit):
    """Polygons (in metres) where the smoothed concentration is at least `level`."""
    tr, crs, w, h = meta
    xs = tr.c + (np.arange(w) + 0.5) * tr.a
    ys = tr.f + (np.arange(h) + 0.5) * tr.e
    z = np.ma.masked_invalid(smooth(a, SMOOTH_CELLS))
    gen = contourpy.contour_generator(xs, ys, z, fill_type=contourpy.FillType.OuterOffset)
    points, offsets = gen.filled(level, 1000)
    polys = []
    for p, o in zip(points, offsets):
        rings = [p[o[i]:o[i + 1]] for i in range(len(o) - 1)]
        poly = make_valid(Polygon(rings[0], rings[1:]))
        if poly.area >= MIN_AREA_KM2 * 1e6:
            polys.append(poly)
    if not polys:
        return []
    g = unary_union(polys).simplify(SIMPLIFY_KM * 1000, preserve_topology=True).difference(slit)
    parts = list(g.geoms) if hasattr(g, "geoms") else [g]
    return [p for p in parts if p.geom_type == "Polygon" and p.area >= MIN_AREA_KM2 * 1e6]


def to_lonlat(polys, crs):
    """Polygons in longitude/latitude, cut at POLE_CUT_LAT, as one GeoJSON MultiPolygon."""
    cut = box(-180, -90, 180, POLE_CUT_LAT)
    out = []
    for poly in polys:
        poly = segmentize(poly, STEP_KM * 1000)  # a straight edge on the polar grid is not straight in longitude/latitude
        def ring(r):
            x, y = r.xy
            lon, lat = transform(crs, "EPSG:4326", list(x), list(y))
            return list(zip(lon, lat))
        p = make_valid(Polygon(ring(poly.exterior), [ring(i) for i in poly.interiors])).intersection(cut)
        for q in (p.geoms if hasattr(p, "geoms") else [p]):
            if q.geom_type == "Polygon" and not q.is_empty:
                out.append(q)
    if not out:
        return None
    coords = []
    for q in out:
        rings = [q.exterior] + list(q.interiors)
        coords.append([[[round(x, 2), round(y, 2)] for x, y in r.coords] for r in rings])
    return {"type": "MultiPolygon", "coordinates": coords}


def cap_level(a, meta):
    """Index of the shade for the polar cap: the median ice north of POLE_CUT_LAT (-1 = no ice)."""
    tr, crs, w, h = meta
    (x,), (y,) = transform("EPSG:4326", crs, [0.0], [POLE_CUT_LAT])
    radius = np.hypot(x, y)
    xs = tr.c + (np.arange(w) + 0.5) * tr.a
    ys = tr.f + (np.arange(h) + 0.5) * tr.e
    inside = np.hypot(*np.meshgrid(xs, ys)) <= radius
    median = float(np.nanmedian(a[inside]))
    return max([i for i, lv in enumerate(LEVELS) if median >= lv], default=-1)

## 6. The ship and the ice

For each day, the ship's position at noon is taken from the route, and compared with that day's ice: the distance to the nearest cell with at least `ICE_MIN` percent ice, and the concentration under the ship.

In [ ]:
track = np.array(route["arctic"]["path"])
track_t = np.array(route["arctic"]["t"])


def ship_at(day):
    lon = np.interp(day, track_t, track[:, 0])
    lat = np.interp(day, track_t, track[:, 1])
    return ((lon + 180) % 360) - 180, lat


def ship_vs_ice(a, meta, day):
    tr, crs, w, h = meta
    lon, lat = ship_at(day)
    if lat < SOUTH_LIMIT:
        return None, None
    (x,), (y,) = transform("EPSG:4326", crs, [lon], [lat])
    iy, ix = np.nonzero(a >= ICE_MIN)
    if len(ix) == 0:
        return None, 0
    ice_x = tr.c + (ix + 0.5) * tr.a
    ice_y = tr.f + (iy + 0.5) * tr.e
    dist = float(np.sqrt((ice_x - x) ** 2 + (ice_y - y) ** 2).min() / 1000)
    row = int((y - tr.f) / tr.e)
    col = int((x - tr.c) / tr.a)
    under = a[row, col] if 0 <= row < h and 0 <= col < w else np.nan
    return round(dist), (None if np.isnan(under) else int(under))

## 7. Run all days

Days with no file are skipped and listed.

In [ ]:
features_out = []
rows = []
missing = []
area = None
slit = None
start = time.time()
for k, d in enumerate(dates):
    ymd = d.strftime("%Y%m%d")
    path = raw / FILE.format(ymd=ymd)
    if not path.exists():
        missing.append(ymd)
        continue
    a, meta = read_day(path)
    if area is None:
        area = cell_area_km2(meta)
    if slit is None:
        slit = make_slit(meta[1])
    extent = float(np.nansum(area[a >= ICE_MIN]))
    for b, level in enumerate(LEVELS):
        geom = to_lonlat(ice_shapes(a, meta, level, slit), meta[1])
        if geom is not None:
            features_out.append({"type": "Feature", "properties": {"d": k, "b": b}, "geometry": geom})
    dist, under = ship_vs_ice(a, meta, k + 0.5)
    rows.append({"day": k, "date": d.date().isoformat(), "extent_km2": round(extent), "ship_to_ice_km": dist, "ice_at_ship_pct": under, "cap_level": cap_level(a, meta)})
    print(f"\r{k + 1}/{len(dates)} days, {time.time() - start:4.0f} s", end="")
print()
days = pd.DataFrame(rows)
print(f"{len(days)} days processed, {len(missing)} missing: {missing}")
days

## 8. What the ice says about the voyage

The two passages through the Northern Sea Route come from `00route`. For each, the closest the ship came to the ice (on our reconstructed track), and the Arctic ice extent at the time.

In [ ]:
summary = []
for p in route["passages"]:
    sel = days[(days["day"] + 0.5 >= p["enter_day"]) & (days["day"] + 0.5 <= p["leave_day"])]
    summary.append({
        "direction": p["direction"],
        "days_with_data": len(sel),
        "closest_to_ice_km": None if sel["ship_to_ice_km"].isna().all() else int(sel["ship_to_ice_km"].min()),
        "median_to_ice_km": None if sel["ship_to_ice_km"].isna().all() else int(sel["ship_to_ice_km"].median()),
        "most_ice_under_ship_pct": None if sel["ice_at_ship_pct"].isna().all() else int(sel["ice_at_ship_pct"].max()),
        "extent_million_km2": round(sel["extent_km2"].mean() / 1e6, 2) if len(sel) else None,
    })
summary_df = pd.DataFrame(summary)
low = days.loc[days["extent_km2"].idxmin()] if len(days) else None
if low is not None:
    print(f"Smallest extent in this period: {low['extent_km2'] / 1e6:.2f} million km2 on {low['date']}")
summary_df

## 9. Check map (polar view of one day, for a quick look only)

In [ ]:
CHECK_DATE = "2026-09-05"  # <- any date of the voyage with a file
k = int((pd.Timestamp(CHECK_DATE) - dates[0]).days)
a, meta = read_day(raw / FILE.format(ymd=pd.Timestamp(CHECK_DATE).strftime("%Y%m%d")))
tr, crs, w, h = meta
fig, ax = plt.subplots(figsize=(7, 8))
ax.imshow(np.where(np.isnan(a), -20, a), cmap="Blues_r", vmin=-20, vmax=100,
          extent=[tr.c, tr.c + w * tr.a, tr.f + h * tr.e, tr.f])  # VISUAL: land shows darkest
lon = ((track[:, 0] + 180) % 360) - 180
keep = track[:, 1] > SOUTH_LIMIT
xs, ys = transform("EPSG:4326", crs, lon[keep].tolist(), track[keep, 1].tolist())
ax.plot(xs, ys, ".", ms=1, color="#d97706")  # VISUAL: the route
(sx,), (sy,) = transform("EPSG:4326", crs, *[[v] for v in ship_at(k + 0.5)])
ax.plot(sx, sy, "o", ms=8, color="#dc2626")  # VISUAL: the ship that day
ax.set_xlim(-3000000, 3000000)
ax.set_ylim(-3000000, 3500000)
ax.set_title(f"Sea ice on {CHECK_DATE} and the ship (red)")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

## 10. Images for the 3D page

The 3D globe (Cesium) cannot fill shapes that go all the way round the pole, so it gets the ice as a picture instead: for each day, the smoothed concentration is moved onto a longitude/latitude grid of `IMG_DEG` degrees and painted in the three shades, with everything else transparent. Cesium drapes the picture over the globe right up to the pole, so it needs neither the polar disc nor the slit.

In [ ]:
img_dir = Path(IMG_DIR)
img_dir.mkdir(parents=True, exist_ok=True)
W, H = int(round(360 / IMG_DEG)), int(round((90 - IMG_SOUTH) / IMG_DEG))
start = time.time()
sizes = []
for k, d in enumerate(dates):
    ymd = d.strftime("%Y%m%d")
    path = raw / FILE.format(ymd=ymd)
    if not path.exists():
        continue
    a, meta = read_day(path)
    tr, crs, w, h = meta
    grid = np.full((H, W), np.nan, dtype="float32")
    reproject(smooth(a, SMOOTH_CELLS).astype("float32"), grid, src_transform=tr, src_crs=crs, src_nodata=np.nan,
              dst_transform=from_origin(-180, 90, IMG_DEG, IMG_DEG), dst_crs="EPSG:4326", dst_nodata=np.nan,
              resampling=Resampling.bilinear)
    rgba = np.zeros((H, W, 4), dtype="uint8")  # fully transparent
    for level, color in zip(LEVELS, IMG_COLORS):
        rgba[grid >= level] = color
    out_png = img_dir / f"{ymd}.png"
    Image.fromarray(rgba, "RGBA").save(out_png, optimize=True)
    sizes.append(out_png.stat().st_size)
    print(f"\r{k + 1}/{len(dates)} images, {time.time() - start:4.0f} s", end="")
print()
print(f"{len(sizes)} images of {W} x {H} pixels, {sum(sizes) / 1e6:.1f} MB in all")

## 11. Write `seaice.json` and `seaice_days.json`

`ice` holds the ice shapes of every day, each tagged with its day `d` and shade `b` (0 = at least 15 percent, 1 = at least 50, 2 = at least 80). `days` holds the daily numbers, `summary` the passage figures.

In [ ]:
days_json = days.astype(object).where(days.notna(), None)  # empty values become null in the file
out = {
    "source": "AMSR2 sea ice concentration, ASI algorithm (Spreen, Kaleschke and Heygster 2008), University of Bremen",
    "source_url": "https://seaice.uni-bremen.de",
    "levels": LEVELS,
    "pole_cut_lat": POLE_CUT_LAT,
    "image_dir": "data/ice",
    "image_bounds": [-180, IMG_SOUTH, 180, 90],
    "ice_min": ICE_MIN,
    "days": days_json.to_dict(orient="records"),
    "summary": summary_df.to_dict(orient="records"),
    "missing": missing,
    "ice": {"type": "FeatureCollection", "features": features_out},
}
Path(OUT_FILE).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_FILE, "w") as f:
    json.dump(out, f, separators=(",", ":"))
print(f"Saved {OUT_FILE}: {Path(OUT_FILE).stat().st_size / 1e6:.1f} MB, {len(features_out)} ice shapes")

small = {k: v for k, v in out.items() if k != "ice"}  # the same without the shapes
with open(DAYS_FILE, "w") as f:
    json.dump(small, f, separators=(",", ":"))
print(f"Saved {DAYS_FILE}: {Path(DAYS_FILE).stat().st_size / 1e3:.0f} KB")

___
**END**